# RLVR guided by the filtered-CoT residual stream — 3 pipelines to iterate

**Premise (established).** Grafting the *attention-filtered* critical CoT latents at layer L=18 and generating
gives **97%** (CoT ~99%), sparse (top-8 → 96%), content-specific (mismatched-donor control is negative).
So the K≈16 critical latents are a **causally-sufficient, verified guidance signal**.

**Problem (established).** *Predicting/distilling* those latents from the query (frozen predictor 4.5%, VQ 5.6%,
LoRA-distill 19.3%) does not generalize — supervised L2 to a static target can't do the serial computation.

**Idea.** Use RL-with-verifiable-rewards (RLVR; reward = GSM8K answer correctness, exact and free) to *search* for
the reasoning, and use the filtered critical residuals as the **guidance signal** that makes that search tractable
(warm prior, causal critic, expert scaffold). Below: 3 distinct ways to inject that signal.

Shared notation: `h_L(pos)` = policy residual at layer L, position `pos`. `Z* = {z*_1..z*_K}` = teacher's
attention-filtered critical latents for a problem (available at train time from the gold CoT; cached in
`latents_L18_K16.pt` / `targets_cache/`). `r_verify ∈ {0,1}` = final-answer correctness.

 *Ideas*. 
 Neologism for reasoning instead. We take 4 rarely used token from the tokenizer's vocabulary, then we train the model to use these $K$ new tokens to carry out its reasoning, we want to see if this beats the naive filler token based reasoning. W

## Comparison map

In [ ]:
import pandas as pd
rows = [
 ["P2 latent-action RLVR","predictor emits K latents→graft","KL/L2 to Z* as reference prior, anneal β","r_verify on grafted gen","predictor(+LoRA)","frozen base (97% graft)","REINFORCE/GRPO on latents","continuous-action variance; Z* is warm start not target"],
 ["P3 residual critic PRM","token/latent policy","V(h_L)→P(correct), labeled by graft-then-check","ΔV process reward + r_verify","critic then policy","base for labeling","PPO w/ latent critic","reward hacking → KL leash + hold-out verify"],
 ["P4 graft-expert iteration","token CoT policy","graft TRUE Z* → forced ~97% expert rollouts","r_verify; expert traj in buffer","policy","graft = demonstrator only","expert-iter / RLVR+SFT","scaffold-removal shift → anneal p, KL to scaffolded π"],
]
pd.set_option("display.max_colwidth", 60)
print(pd.DataFrame(rows, columns=["pipeline","policy","residual guidance","reward","trains","frozen","RL algo","main risk"]).to_string(index=False))

                 pipeline                          policy                                  residual guidance                          reward             trains                    frozen                   RL algo                                                                 main risk
  P1 residual-shaped RLVR                token CoT policy          dense potential Φ=align(h_L, Z*), F=γΦ'−Φ              r_verify + shaping             policy         teacher extractor                  GRPO/PPO alignment is positional-soft; keep Φ a pure state fn (shaping invariance)
    P2 latent-action RLVR predictor emits K latents→graft           KL/L2 to Z* as reference prior, anneal β         r_verify on grafted gen   predictor(+LoRA)   frozen base (97% graft) REINFORCE/GRPO on latents                   continuous-action variance; Z* is warm start not target
   P3 residual critic PRM             token/latent policy     V(h_L)→P(correct), labeled by graft-then-check    ΔV process reward + r_verify c

## Shared building blocks
These are the reusable pieces the remaining pipelines call. `Z*` extraction + grafting already exist and are verified
(`latent_predict.extract`, the graft hook in `graft_generate.py`).

In [ ]:
import torch, torch.nn.functional as F

# --- graft hook factory (verified causal injection at layer L) ---
def graft_hook(latents, positions):
    def hook(mod, inp, out, _l=latents, _p=positions):
        o = out[0] if isinstance(out, tuple) else out
        if o.shape[1] >= max(_p) + 1:
            o = o.clone(); o[:, _p, :] = _l.to(o.dtype)
        return (o,) + tuple(out[1:]) if isinstance(out, tuple) else o
    return hook
print("block defined: graft_hook")

blocks defined: residual_alignment / potential_shaping / graft_hook


P1 reward defined


## P2 · Latent-action RLVR  ← most direct fix for the generalization wall
The **policy is the latent predictor**: it emits K continuous thoughts, we graft them into the frozen model
(97% graft guarantee holds) and generate. Reward = `r_verify`. The critical latents `Z*` act as a **reference
prior**, not a fixed target: objective = `E[r] − β·‖z − Z*‖²` with β annealed high→low, so RL *searches around*
the causally-valid region instead of regressing to a static point (which we know fails at 4.5%).

In [4]:
from latent_predict import Predictor            # reuse the cross-attn predictor
def p2_step(model, pred, item, L=18, K=16, beta=0.1, n_samp=8, sigma=0.05, fid=None, tok=None):
    # policy = Gaussian around predictor mean; sample n_samp latent-actions, graft, reward, REINFORCE
    mem, mask = item["qh"].unsqueeze(0).cuda(), None
    mu = pred(mem, torch.zeros(1, item["qh"].size(0), dtype=torch.bool).cuda())[0]     # (K,d)
    logps, rews = [], []
    for _ in range(n_samp):
        eps = torch.randn_like(mu) * sigma
        z = mu + eps
        logp = (-(eps**2).sum() / (2*sigma**2))                       # Gaussian logprob (up to const)
        r = graft_generate_reward(model, tok, item, z, L, K, fid)     # graft z, generate, r_verify
        logps.append(logp); rews.append(r)
    R = torch.tensor(rews); A = (R - R.mean())                        # GRPO-style group baseline
    pg = -(torch.stack(logps) * A.cuda()).mean()                      # policy gradient
    reg = beta * F.mse_loss(mu, item["tgt"].cuda())                   # residual reference prior (annealed)
    return pg + reg, R.mean().item()
# graft_generate_reward = graft z at layer L into [q][K fillers][####], greedy-gen, compare last_int==gold.
# Ablations: beta schedule (0.5→0), sigma, n_samp; warm-start pred from latent_predict_l2.json weights.
print("P2 step defined (RL-finetune the latent predictor with residual prior)")

P2 step defined (RL-finetune the latent predictor with residual prior)


## P3 · Residual critic (process reward model from grafting)
RLVR's reward is terminal, but grafting says the layer-L latents are the causal bottleneck — so score *them*.
Train a value head `V(h_L)→P(correct)` with **free, causal labels**: graft a candidate residual, generate,
check. `Z*` gives positives; random/mismatched residuals (the grafting control, verified negative) give
negatives. Then use ΔV as a **dense process reward** / PPO critic in latent space.

In [5]:
import torch.nn as nn
class ResidualCritic(nn.Module):
    def __init__(self, d): super().__init__(); self.f = nn.Sequential(nn.Linear(d,d), nn.GELU(), nn.Linear(d,1))
    def forward(self, h_L): return self.f(h_L).squeeze(-1)          # logit P(final correct | this latent)

def build_critic_labels(model, tok, items, L, K, fid):
    # positives: graft Z* -> correct? ; negatives: graft shuffled/other-problem Z' -> correct?
    X, y = [], []
    for it in items:
        X.append(it["tgt"]);            y.append(graft_correct(model,tok,it,it["tgt"],L,K,fid))     # ~1
        other = items[(items.index(it)+7) % len(items)]["tgt"]
        X.append(other);                y.append(graft_correct(model,tok,it,other,L,K,fid))         # ~0 (control)
    return X, y
# Use: process reward r_t = V(h_L(step t+1)) - V(h_L(step t)); or PPO advantage from V. KL-leash policy to base.
print("P3 residual critic defined")

P3 residual critic defined


## P4 · Grafting-as-expert iteration (residual scaffold curriculum)
Cures RL cold-start without any gold-CoT *text*: during rollouts, with prob `p` **graft the true `Z*`** to force
a guaranteed-good (~97%) trajectory, and add those residual-induced experts to the buffer as off-policy positives
(expert iteration). Anneal the scaffold — graft prob `p` and/or strength `α` (`h←(1−α)h+α·z*`) from 1→0 as the
policy's *unaided* accuracy rises. The scaffold is a demonstrator only; at test time nothing is grafted.

In [6]:
def p4_rollout(model, tok, item, policy, p_graft, alpha, L, K, fid):
    use = (hash((item['gold_str'], round(p_graft,2))) % 100) / 100.0 < p_graft   # det. pseudo-rand (no Random in nb)
    if use:
        h = graft_hook((1-alpha)*0 + alpha*item["tgt"].cuda(), list(range(len(item["q"]), len(item["q"])+K)))
        handle = model.model.layers[L].register_forward_hook(h)
        try:    traj = policy.generate_traj(item)          # forced expert (~97% correct)
        finally: handle.remove()
        traj.is_expert = True
    else:
        traj = policy.generate_traj(item); traj.is_expert = False
    return traj                                            # RLVR loss weights experts as off-policy positives
# Curriculum: p_graft, alpha <- schedule(step, unaided_acc). Keep KL(policy || scaffolded_policy) leashed.
print("P4 graft-expert rollout defined")

P4 graft-expert rollout defined


P5 concise+faithful reward defined


## What to try first (ranked)
1. **P2** — most directly attacks the wall we hit: RL lets the predictor *search* around the causal `Z*` region instead of regressing to it. Warm-start from `latent_predict_l2.json` (running now).
2. **P4** — strongest cold-start cure; the residual scaffold gives guaranteed-good trajectories with no gold-CoT text.
3. **P3** — a frozen residual critic can provide a dense signal without aligning a moving policy to fixed residual coordinates.

Baselines to beat: filler-only 4.3%, LoRA-distill **19.3%**, full-FT distill (running). Target: CoT ~99%.
All 3 reuse the verified `Z*` cache + graft hook — no new extraction needed.